# Vérification / renumérotation des actes à partir des zones (JJ167+)

Pour les registres à partir de JJ167, le tableau d'actes ne donne plus systématiquement
le folio de début de chaque acte. Ce notebook reconstitue une numérotation de contrôle
à partir des zones **AI** et **AC** (dans l'ordre de lecture : folio, puis position verticale),
la confronte aux repères connus du tableau d'actes, et produit un rapport HTML interactif
permettant de pointer/corriger la numérotation au fil du travail.

Ce notebook est autonome : il recharge et reconstruit `df_images`, `df_actes`, `df_zones`
à partir des fichiers sources (mêmes fonctions que le notebook principal
`JJ100-139-versImportLabelStudio2.ipynb`), puis applique les fonctions de vérification.


# Paramètres

In [ ]:
import os
import re

corpus = 'JJ167-JJ195'   # à adapter au corpus traité

# Chemins des fichiers sources
INPUT_ZONES  = f"../List-of-zones/Himanis_Seg_Actes_1200pxmin_{corpus}_labelstudio.csv"       # fichier 1 : zones YOLO
INPUT_ACTES  = "../List-of-acts/Acts-JJ96-JJ195_enriched_Locus-20260521.xlsm"                  # fichier 2 : liste des actes
INPUT_IMAGES = f"../List-of-images/{corpus}_image_data.csv"    # fichier 3 : images téléchargées

# Dossier de sortie
OUT_DIR = f"{corpus}/output"
os.makedirs(OUT_DIR, exist_ok=True)

# Labels de zones attendus par le modèle d'acte
LABELS_ATTENDUS = {'AC', 'AI', 'AM', 'AF', 'NIA', 'Table'}


def parse_corpus(corpus):
    match = re.match(r'([A-Z]+)(\d+)-([A-Z]+)(\d+)', corpus)
    if not match:
        raise ValueError(f"Format inattendu : {corpus}")
    prefix = match.group(1)
    start, end = int(match.group(2)), int(match.group(4))
    return {f"{prefix}{i}" for i in range(start, end + 1)}

REGISTRES_CIBLES = parse_corpus(corpus)
print(REGISTRES_CIBLES)

# Séparateur CSV des fichiers sources ('\t' = tabulation)
SEP = '\t'


# Imports et fonctions utilitaires (identiques au notebook principal)

In [ ]:
import os
import re
import ast
import json
import warnings
import pandas as pd
import numpy as np
from collections import defaultdict


pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)


def normalize_path(p):
    """Normalise les séparateurs Windows/Unix et renvoie le basename."""
    return os.path.basename(str(p).replace("\\\\", "/"))


def extract_register(folder_path):
    """
    Extrait le numéro de registre normalisé depuis un chemin.
    'Paris_Archives_Nationales_JJ096' → 'JJ96' (sans zéro initial).
    Cible le DERNIER composant du chemin contenant JJ + chiffres.
    """
    parts = str(folder_path).replace("\\\\", "/").split("/")
    for part in reversed(parts):
        m = re.search(r'JJ0*(\d+)$', part)
        if m:
            return f"JJ{m.group(1)}"
    return None


def normalize_folio(raw):
    """
    Normalise un label de folio pour jointure.
      '6'        -> '6r'
      '12v'      -> '12v'
      '1r'       -> '1r'
      '103bis'   -> '103bisr'
      '103bis v' -> '103bisv'
      '103 bis recto' -> '103bisr'
      '103 bis verso' -> '103bisv'
      'plat supérieur' -> 'plat supérieur'
    """
    s = str(raw).strip().lower()

    if s in ('vacat', 'vacatr', 'vacatv'):
        return None   # sera logué comme "sans folio" -> ignoré proprement

    if not s or s == 'nan':
        return None

    # Normaliser les variantes textuelles de recto/verso
    s = s.replace('recto', 'r').replace('verso', 'v')

    # Supprimer les espaces autour de 'bis' et avant r/v
    s = re.sub(r'\s+bis\s*', 'bis', s)
    s = re.sub(r'\s+([rv])$', r'\1', s)

    # Cas standard : chiffres + optionnel 'bis' + r ou v
    if re.match(r'^\d+(?:bis)?[rv]$', s):
        return s

    # Chiffres + optionnel 'bis' sans suffixe -> recto par défaut
    if re.match(r'^\d+(?:bis)?$', s):
        return s + 'r'

    return s


def parse_abs_coords(coord_str):
    """Parse '1,23,3866,6279' -> [x, y, w, h] ou None si invalide."""
    try:
        parts = [int(v) for v in str(coord_str).split(',')]
        if len(parts) == 4:
            return parts
    except Exception:
        pass
    return None

def parse_image_stem(image_path):
    stem = normalize_path(image_path).rsplit('.', 1)[0]  # retire .jpg
    parts = stem.split('_')
    volume = parts[-2]               # 'JJ096'
    folio_sort_key = int(parts[-1])  # 100
    return volume, folio_sort_key


def safe_to_int(series):
    return (
        series
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
        .round()
        .astype(int)
    )


def strip_leading_zeros(folio_norm):
    """'001r' -> '1r', '012bisv' -> '12bisv' (règle spécifique JJ126, réutilisable ailleurs)."""
    if not folio_norm:
        return folio_norm
    m = re.match(r'^0*(\d+)(bis)?([rv])$', folio_norm)
    if m:
        return f"{m.group(1)}{m.group(2) or ''}{m.group(3)}"
    return folio_norm


# Chargement des fichiers CSV sources

## Table `images.csv`

In [ ]:
skipped = {}

with warnings.catch_warnings(record=True) as w_images:
    warnings.simplefilter("always")
    df_images_raw = pd.read_csv(INPUT_IMAGES, sep=',', dtype=str,
                             low_memory=False, on_bad_lines='warn', encoding='utf-8')
    skipped['images'] = len(w_images)

print(f"Images chargées : {len(df_images_raw):>6} lignes  ({skipped['images']} ligne(s) ignorée(s))")

df_images = df_images_raw[[
    'manifestURL', 'canvasId', 'urlImage', 'imageLabel',
    'imageFileName', 'imageWidthAsDownloaded', 'imageHeightAsDownloaded',
    'urlResizedImage', 'ResizedImageWidthAsDownloaded',
    'ResizedImageHeightAsDownloaded', 'folderPath'
]].copy()

df_images['image_id']       = range(1, len(df_images) + 1)
df_images['image_filename'] = df_images['imageFileName'].apply(normalize_path)
df_images[['volume', 'folio_sort_key']] = df_images['image_filename'].apply(
    lambda p: pd.Series(parse_image_stem(p))
    )
df_images['register']       = df_images['folderPath'].apply(extract_register)
df_images['folio_norm']     = df_images['imageLabel'].apply(normalize_folio)

df_images.rename(columns={
    'manifestURL':                    'manifest_url',
    'canvasId':                       'canvas_id',
    'urlImage':                       'url_full',
    'imageLabel':                     'folio_label',
    'imageWidthAsDownloaded':         'width_px',
    'imageHeightAsDownloaded':        'height_px',
    'urlResizedImage':                'url_resized',
    'ResizedImageWidthAsDownloaded':  'resized_width_px',
    'ResizedImageHeightAsDownloaded': 'resized_height_px',
}, inplace=True)

df_images.drop(columns=['imageFileName', 'folderPath'], inplace=True)

IMAGES_COLS = [
    'image_id', 'register', 'image_filename', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'width_px', 'height_px', 'url_full', 'url_resized',
    'resized_width_px', 'resized_height_px', 'manifest_url', 'canvas_id',
]
df_images = df_images[IMAGES_COLS]

df_images.to_csv(os.path.join(OUT_DIR, "images.csv"), index=False, sep=SEP)
print(f"images.csv -> {len(df_images)} lignes, {len(df_images.columns)} colonnes")

# Index (volume, folio_sort_key) -> [image_id, ...] (ordre séquentiel préservé)
img_by_folio = defaultdict(list)
for _, row in df_images.iterrows():
    if pd.notna(row['volume']) and pd.notna(row['folio_sort_key']):
        img_by_folio[(row['volume'], int(row['folio_sort_key']))].append(row['image_id'])

images_by_register = defaultdict(list)
for _, row in df_images.iterrows():
    if row['register']:
        images_by_register[row['register']].append(row['image_id'])

print(f"Index img_by_folio    : {len(img_by_folio)} clés (volume, folio_sort_key)")
print(f"Registres distincts   : {list(images_by_register.keys())}")

df_images.head(3)


## Table `actes.csv`

In [ ]:
skipped = {}

with warnings.catch_warnings(record=True) as w_actes:
    warnings.simplefilter("always")
    df_actes_raw = pd.read_excel(INPUT_ACTES)
    skipped['actes'] = len(w_actes)

print(f"Actes  chargés  : {len(df_actes_raw):>6} lignes  ({skipped['actes']} ligne(s) ignorée(s))")

df_actes = df_actes_raw.copy()

df_actes.rename(columns={
    'ID-temporaire':        'acte_id',
    'Register':             'volume',
    'Act_number':           'act_number',
    'Nvelle numérotation':  'new_numbering',
    'Folio Number ou page': 'folio_raw',
    'vérif':                'verified',
    'Note':                 'note',
}, inplace=True)

df_actes = df_actes[df_actes['volume'].isin(REGISTRES_CIBLES)].copy()
print(f"Actes après filtre {corpus} : {len(df_actes)} lignes")
print(df_actes['volume'].value_counts().sort_index())

df_actes['folio_norm'] = df_actes['folio_raw'].apply(normalize_folio)

# Table de correspondance folio_norm -> folio_sort_key par registre
folio_to_sortkey = (df_images[['volume', 'folio_norm', 'folio_sort_key']]
                    .drop_duplicates(subset=['volume', 'folio_norm'])
                    .reset_index(drop=True))

df_actes = df_actes.merge(
    folio_to_sortkey,
    on=['volume', 'folio_norm'],
    how='left'
)

unmatched = df_actes[df_actes['folio_sort_key'].isna()]
if not unmatched.empty:
    print(f"AVERTISSEMENT : {len(unmatched)} acte(s) sans folio_sort_key :")
    print(unmatched[['acte_id', 'volume', 'folio_norm']].to_string(index=False))
else:
    print("Tous les folios ont été matchés.")

# Aplatir les concordances : chaque inventaire source donne 3 colonnes
concordance_cols = [c for c in df_actes.columns if '.xml_' in c]
inventory_sources = defaultdict(list)
for col in concordance_cols:
    src = col.split('.xml_')[0] + '.xml'
    inventory_sources[src].append(col)

for src, cols in inventory_sources.items():
    short = re.sub(r'^Paris_AN_JJ_inventaire_', '', src.replace('.xml', ''))
    short = short.replace('Guerin_tome1-tome12', 'Guerin')
    for suffix, key in [('_Act_number', 'act'), ('_Head', 'head'), ('_Locus', 'locus')]:
        col = next((c for c in cols if c.endswith(suffix)), None)
        if col:
            df_actes[f'conc_{short}_{key}'] = df_actes[col].fillna('')

BASE_COLS = ['acte_id', 'volume', 'act_number', 'new_numbering',
             'folio_raw', 'folio_norm', 'verified', 'note']
CONC_COLS = [c for c in df_actes.columns if c.startswith('conc_')]
df_actes_out = df_actes[BASE_COLS + CONC_COLS].copy()

df_actes_out.to_csv(os.path.join(OUT_DIR, "actes.csv"), index=False, sep=SEP)
print(f"actes.csv -> {len(df_actes_out)} lignes, {len(df_actes_out.columns)} colonnes")
df_actes_out.head(8)


## Table `zones.csv`

In [ ]:
skipped = {}

with warnings.catch_warnings(record=True) as w_zones:
    warnings.simplefilter("always")
    df_zones_raw = pd.read_csv(INPUT_ZONES, sep=',', dtype=str,
                               low_memory=False, on_bad_lines='warn')
    skipped['zones'] = len(w_zones)

# Normalisation des noms de colonnes -> le reste du code utilise toujours
# 'volume', 'folio_sort_key', peu importe le corpus
df_zones_raw = df_zones_raw.rename(columns={
    'registre': 'volume',
    'ordre':    'folio_sort_key',
})

def parse_labelstudio_rects(label_str):
    if pd.isna(label_str):
        return []
    if not str(label_str).strip():
        return []
    try:
        data = json.loads(label_str)
    except Exception:
        try:
            data = ast.literal_eval(label_str)
        except Exception:
            return []
    if not isinstance(data, list):
        return []

    rows = []
    for r in data:
        if 'rectanglelabels' not in r:
            continue
        rows.append({
            'class_name': r['rectanglelabels'][0],
            'x_pct': r['x'],
            'y_pct': r['y'],
            'w_pct': r['width'],
            'h_pct': r['height'],
            'image_width_px': r['original_width'],
            'image_height_px': r['original_height'],
        })
    return rows

# Supprime lignes entièrement vides ou sans annotation
df_zones_raw = df_zones_raw.dropna(how='all')
df_zones_raw = df_zones_raw.dropna(subset=['label'])
df_zones_raw = df_zones_raw[
    df_zones_raw['label'].astype(str).str.strip() != ''
]
print(f"Zones importées : {len(df_zones_raw)}")

df_zones_raw['url_image_full'] = df_zones_raw['image']

df_zones_raw['volume'] = df_zones_raw['volume'].apply(
    lambda r: re.sub(r'JJ(\d+)', lambda m: f"JJ{int(m.group(1)):03d}", str(r))
)
df_zones_raw['folio_sort_key'] = pd.to_numeric(df_zones_raw['folio_sort_key'], errors='coerce')

rows = []
for _, r in df_zones_raw.iterrows():
    annotations = parse_labelstudio_rects(r['label'])
    for ann in annotations:
        rows.append({
            'url_image_full':   r['url_image_full'],
            'image_path':       r['image_path'],
            'image_filename':   r['image'],
            'volume':           r['volume'],
            'folio_sort_key':   r['folio_sort_key'],
            'class_name':       ann['class_name'],
            'x_pct':            ann['x_pct'],
            'y_pct':            ann['y_pct'],
            'w_pct':            ann['w_pct'],
            'h_pct':            ann['h_pct'],
            'image_width_px':   ann['image_width_px'],
            'image_height_px':  ann['image_height_px'],
        })

df_zones = pd.DataFrame(rows)

df_zones['abs_x'] = safe_to_int(df_zones['x_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_y'] = safe_to_int(df_zones['y_pct'] / 100.0 * df_zones['image_height_px'])
df_zones['abs_w'] = safe_to_int(df_zones['w_pct'] / 100.0 * df_zones['image_width_px'])
df_zones['abs_h'] = safe_to_int(df_zones['h_pct'] / 100.0 * df_zones['image_height_px'])

df_zones = df_zones.sort_values(
    by=['volume', 'folio_sort_key', 'abs_y', 'abs_x'],
    ascending=[True, True, True, True]
    )

img_lookup = df_images[['image_id', 'volume', 'folio_sort_key',
                         'folio_label', 'folio_norm']].copy()

df_zones = df_zones.merge(
    img_lookup,
    on=['volume', 'folio_sort_key'],
    how='left'
)

df_zones['zone_id'] = np.arange(1, len(df_zones) + 1)

CLASS_MAP = {
    'AC': 0,
    'AI': 1,
    'AF': 2,
    'AM': 3,
    'NIA': 4,
    'Table': 5,
}

df_zones['class_id'] = df_zones['class_name'].map(CLASS_MAP)

ZONES_COLS = [
    'zone_id', 'image_id', 'volume', 'folio_sort_key',
    'folio_label', 'folio_norm',
    'class_id', 'class_name',
    'abs_x', 'abs_y', 'abs_w', 'abs_h',
    'x_pct', 'y_pct', 'w_pct', 'h_pct',
    'url_image_full', 'image_filename',
    'image_width_px', 'image_height_px',
]
df_zones = df_zones[ZONES_COLS]

df_zones.to_csv(os.path.join(OUT_DIR, "zones.csv"), index=False, sep=SEP)
print(f"zones.csv -> {len(df_zones)} lignes, {len(df_zones.columns)} colonnes")
df_zones.head(10)


# Fonctions de vérification / renumérotation

## Construction de l'URL IIIF d'une région

In [ ]:
def build_iiif_region_url(url_full, x_pct, y_pct, w_pct, h_pct):
    """
    Construit l'URL IIIF d'une région à partir de l'URL de l'image pleine,
    en utilisant directement les pourcentages Label Studio (région IIIF 'pct:').
    On évite ainsi tout recalcul en pixels absolus, qui suppose à tort que
    'original_width'/'original_height' (résolution de l'image annotée dans
    Label Studio) correspond à la résolution de l'image IIIF plein format --
    ce qui n'est pas le cas (ex. image servie en /full/1200,/ alors que le
    plein format est plus grand).
    Structure IIIF : {id}/{region}/{size}/{rotation}/{quality}.{format}
    On remplace uniquement le paramètre 'region' (toujours 'full' en entrée),
    identifié par sa position -- les 3 segments suivants (size/rotation/quality.format)
    sont préservés tels quels.
    """
    region = f"pct:{x_pct},{y_pct},{w_pct},{h_pct}"
    return re.sub(
        r'/full/([^/]+/[^/]+/[^/]+)$',
        f'/{region}/\\1',
        str(url_full)
    )


## Numérotation calculée depuis les zones AI/AC

In [ ]:
def build_zone_sequence(df_zones, volume):
    """
    Séquence des actes calculée depuis les zones AI/AC d'un volume,
    dans l'ordre de lecture (folio, puis position verticale/horizontale sur la page).
    Une zone AI = un acte qui commence sur ce folio (suite éventuelle sur les folios
    suivants via AM/AF, déjà gérée ailleurs dans le pipeline de liaison).
    Une zone AC = un acte complet tenant sur le folio.
    """
    g = (df_zones[(df_zones['volume'] == volume) &
                  (df_zones['class_name'].isin(['AI', 'AC']))]
         .sort_values(['folio_sort_key', 'abs_y', 'abs_x'])
         .reset_index(drop=True)
         .copy())

    g['seq_num']       = range(1, len(g) + 1)
    g['iiif_url_page'] = g['url_image_full']   # image entière (contexte du folio)
    g['iiif_url_zone'] = g.apply(
        lambda r: build_iiif_region_url(r['url_image_full'], r['x_pct'], r['y_pct'],
                                         r['w_pct'], r['h_pct']), axis=1)

    return g[['seq_num', 'zone_id', 'image_id', 'folio_sort_key', 'folio_norm',
              'class_name', 'iiif_url_page', 'iiif_url_zone']]


## Confrontation aux repères connus — plusieurs actes par folio

**Problème traité ici** : un même folio peut porter un AI et un ou plusieurs AC
(plusieurs actes commencent sur la même page). Le tableau d'actes ne donne que
`(act_number, folio_norm)`, sans indication de position sur la page. On apparie donc,
**par folio**, les repères connus (triés par `act_number` croissant) avec les zones
AI/AC détectées sur ce folio (déjà triées par ordre de lecture `abs_y`/`abs_x`),
en supposant que l'ordre de lecture des zones correspond à l'ordre de numérotation
des actes sur la page — hypothèse standard en diplomatique registre.

Si le nombre de repères et le nombre de zones détectées sur un même folio ne
correspondent pas, l'appariement est ambigu : il est signalé par un avertissement
dédié plutôt que d'être forcé.

In [ ]:
def cross_check_sequence(df_seq, df_actes, volume):
    """
    Ajoute act_number_table (repère connu), ecart (table - calculé),
    et un warning quand l'écart change entre deux repères, ou quand
    l'appariement sur un folio est ambigu (plusieurs actes/plusieurs zones).
    """
    df_seq = df_seq.copy()
    df_seq['act_number_table'] = pd.NA
    df_seq['ecart']            = pd.NA
    df_seq['warning']          = ''

    reperes = (df_actes[(df_actes['volume'] == volume) &
                        df_actes['act_number'].notna() &
                        df_actes['folio_norm'].notna()]
               [['act_number', 'folio_norm']].copy())
    reperes['act_number'] = pd.to_numeric(reperes['act_number'], errors='coerce')
    reperes = reperes.dropna(subset=['act_number'])

    # --- Appariement par folio, dans l'ordre ---
    for folio_norm, actes_folio in reperes.groupby('folio_norm'):
        actes_tries = actes_folio.sort_values('act_number')['act_number'].tolist()
        zones_folio_idx = df_seq.index[df_seq['folio_norm'] == folio_norm].tolist()

        if not zones_folio_idx:
            continue

        n_actes = len(actes_tries)
        n_zones = len(zones_folio_idx)

        if n_actes == n_zones:
            for act_num, idx in zip(actes_tries, zones_folio_idx):
                df_seq.loc[idx, 'act_number_table'] = int(act_num)
                df_seq.loc[idx, 'ecart'] = int(act_num) - int(df_seq.loc[idx, 'seq_num'])
        else:
            msg = (f"AMBIGU sur folio {folio_norm} : {n_actes} acte(s) au tableau "
                   f"vs {n_zones} zone(s) AI/AC détectée(s)")
            first_idx = zones_folio_idx[0]
            df_seq.loc[first_idx, 'warning'] = msg
            for act_num, idx in zip(actes_tries, zones_folio_idx):
                df_seq.loc[idx, 'act_number_table'] = int(act_num)
                df_seq.loc[idx, 'ecart'] = int(act_num) - int(df_seq.loc[idx, 'seq_num'])

    # --- Détection des changements d'écart entre repères successifs ---
    last_ecart = None
    for idx, r in df_seq.iterrows():
        if pd.notna(r['ecart']):
            if last_ecart is not None and r['ecart'] != last_ecart:
                existing = df_seq.loc[idx, 'warning']
                new_warn = (f"ecart {last_ecart:+d} -> {r['ecart']:+d} "
                            f"(zone manquante ou en trop entre les deux reperes)")
                df_seq.loc[idx, 'warning'] = (existing + " | " + new_warn) if existing else new_warn
            last_ecart = r['ecart']

    return df_seq


## Export HTML interactif

In [ ]:
import json


def export_interactive_html(df_seq, volume, output_path):
    """
    Exporte un rapport HTML interactif pour vérifier/renuméroter les actes
    d'un volume, à partir du DataFrame produit par cross_check_sequence().
    """
    records = df_seq.copy()
    records['act_number_table'] = records['act_number_table'].astype('Int64')
    records['ecart'] = records['ecart'].astype('Int64')
    records = records.where(records.notna(), None)
    data = records.to_dict('records')

    html = HTML_TEMPLATE.replace('__VOLUME__', volume).replace(
        '__DATA__', json.dumps(data, ensure_ascii=False))

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f"Rapport généré : {output_path}")


HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<title>Vérification des actes — __VOLUME__</title>
<style>
  body { font-family: -apple-system, Segoe UI, Arial, sans-serif; margin: 20px; background:#fafafa; }
  h1 { font-size: 1.3em; }
  .meta { color:#555; margin-bottom: 14px; }
  table { border-collapse: collapse; width: 100%; font-size: 13px; background:#fff; }
  th, td { border: 1px solid #ddd; padding: 5px 8px; text-align: left; vertical-align: middle; }
  th { background: #2c3e50; color: #fff; position: sticky; top: 0; }
  tr.excluded { opacity: 0.35; text-decoration: line-through; }
  tr.warning { background: #fff3cd; }
  tr.ecart-nonzero { background: #f8d7da; }
  tr.anchor { background: #e8f4ea; }
  td.thumb img { height: 40px; border:1px solid #999; cursor: pointer; }
  td.thumb:nth-of-type(5) img { height: 60px; }  /* vignette "folio (contexte)" un peu plus grande */
  input.seq-edit { width: 45px; }
  button { cursor: pointer; }
  .toolbar { margin-bottom: 10px; }
  .warn-text { color:#a15c00; font-weight:600; }
  .lightbox { display:none; position:fixed; inset:0; background:rgba(0,0,0,.85);
              align-items:center; justify-content:center; z-index:10; }
  .lightbox img { max-height:90vh; max-width:90vw; }
</style>
</head>
<body>

<h1>Vérification des actes — __VOLUME__</h1>
<div class="meta">
  Numérotation calculée à partir des zones AI/AC, confrontée aux repères connus du tableau d'actes.
  <br>Coche "exclure" pour retirer une zone erronée (ex. AI mal détectée), ou "insérer avant"
  pour ajouter un acte manquant : la numérotation et les écarts se recalculent automatiquement.
</div>

<div class="toolbar">
  <button onclick="recompute()">Recalculer</button>
  <button onclick="exportCsv()">Exporter CSV (état corrigé)</button>
  <span id="summary" style="margin-left:16px; font-weight:600;"></span>
</div>

<table id="tbl">
  <thead>
    <tr>
      <th>#</th>
      <th>N° calculé</th>
      <th>Folio</th>
      <th>Type</th>
      <th>Folio (contexte)</th>
      <th>Zone</th>
      <th>N° tableau (repère)</th>
      <th>Écart</th>
      <th>Avertissement</th>
      <th>Exclure</th>
      <th>Insérer avant</th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>

<div class="lightbox" id="lightbox" onclick="this.style.display='none'">
  <img id="lightbox-img" src="">
</div>

<script>
let rows = __DATA__.map((r, i) => ({...r, _id: i, _excluded: false, _insertedBefore: false}));

function render() {
  const tbody = document.getElementById('tbody');
  tbody.innerHTML = '';
  rows.forEach((r) => {
    const tr = document.createElement('tr');
    if (r._excluded) tr.classList.add('excluded');
    if (r.ecart !== null && r.ecart !== undefined && r.ecart !== 0) {
      tr.classList.add('ecart-nonzero');
    } else if (r.warning) {
      tr.classList.add('warning');
    }
    if (r.act_number_table !== null && r.act_number_table !== undefined) tr.classList.add('anchor');

    tr.innerHTML = `
      <td>${r._id + 1}</td>
      <td><input class="seq-edit" type="text" value="${r._display_seq ?? ''}" disabled></td>
      <td>${r.folio_norm ?? ''}</td>
      <td>${r.class_name ?? ''}</td>
      <td class="thumb">${r.iiif_url_page ? `<img src="${r.iiif_url_page}" onclick="showLightbox('${r.iiif_url_page}')">` : ''}</td>
      <td class="thumb">${r.iiif_url_zone ? `<img src="${r.iiif_url_zone}" onclick="showLightbox('${r.iiif_url_zone}')">` : ''}</td>
      <td>${r.act_number_table ?? ''}</td>
      <td>${r.ecart ?? ''}</td>
      <td class="warn-text">${r.warning ?? ''}</td>
      <td><input type="checkbox" ${r._excluded ? 'checked' : ''} onchange="toggleExclude(${r._id})"></td>
      <td><button onclick="insertBefore(${r._id})">+ acte</button></td>
    `;
    tbody.appendChild(tr);
  });
}

function toggleExclude(id) {
  const r = rows.find(x => x._id === id);
  r._excluded = !r._excluded;
  recompute();
}

function insertBefore(id) {
  const idx = rows.findIndex(x => x._id === id);
  const blank = {
    zone_id: null, image_id: null, folio_sort_key: rows[idx].folio_sort_key,
    folio_norm: rows[idx].folio_norm, class_name: '(inséré manuellement)',
    iiif_url_page: null, iiif_url_zone: null, act_number_table: null, ecart: null, warning: '',
    _id: -1, _excluded: false
  };
  rows.splice(idx, 0, blank);
  rows.forEach((r, i) => r._id = i);
  recompute();
}

function recompute() {
  let seq = 0;
  let lastEcart = null;
  let warnCount = 0;

  rows.forEach(r => {
    if (r._excluded) { r._display_seq = '—'; return; }
    seq += 1;
    r._display_seq = seq;
    r.warning = '';

    if (r.act_number_table !== null && r.act_number_table !== undefined) {
      r.ecart = r.act_number_table - seq;
      if (lastEcart !== null && r.ecart !== lastEcart) {
        r.warning = `⚠ écart ${lastEcart >= 0 ? '+' + lastEcart : lastEcart} → ${r.ecart >= 0 ? '+' + r.ecart : r.ecart}`;
        warnCount += 1;
      }
      lastEcart = r.ecart;
    }
  });

  document.getElementById('summary').textContent =
    `${rows.filter(r => !r._excluded).length} actes actifs — ${warnCount} avertissement(s)`;
  render();
}

function showLightbox(url) {
  document.getElementById('lightbox-img').src = url;
  document.getElementById('lightbox').style.display = 'flex';
}

function exportCsv() {
  const header = ['seq_num','folio_norm','class_name','iiif_url_page','iiif_url_zone','act_number_table','ecart','warning','excluded'];
  const lines = [header.join(';')];
  rows.forEach(r => {
    lines.push([
      r._display_seq, r.folio_norm ?? '', r.class_name ?? '', r.iiif_url_page ?? '', r.iiif_url_zone ?? '',
      r.act_number_table ?? '', r.ecart ?? '', (r.warning ?? '').replace(/;/g,','), r._excluded
    ].join(';'));
  });
  const blob = new Blob([lines.join('\\n')], {type: 'text/csv;charset=utf-8;'});
  const a = document.createElement('a');
  a.href = URL.createObjectURL(blob);
  a.download = 'verification_actes___VOLUME__.csv';
  a.click();
}

recompute();
</script>
</body>
</html>
"""


# Exécution — un volume

In [ ]:
VOLUME = sorted(REGISTRES_CIBLES)[0]
OUT_HTML_DIR = OUT_DIR

df_seq = build_zone_sequence(df_zones, VOLUME)
df_seq = cross_check_sequence(df_seq, df_actes_out, VOLUME)

print(f"{VOLUME} : {len(df_seq)} actes détectés (AI/AC), "
      f"{df_seq['act_number_table'].notna().sum()} repère(s) connu(s), "
      f"{(df_seq['warning'] != '').sum()} avertissement(s)")

export_interactive_html(df_seq, VOLUME,
                         os.path.join(OUT_HTML_DIR, f'verification_actes_{VOLUME}.html'))

df_seq.head(20)


# Exécution — tous les volumes du corpus

In [ ]:
rapports = {}
for vol in sorted(REGISTRES_CIBLES):
    seq = build_zone_sequence(df_zones, vol)
    if seq.empty:
        continue
    seq = cross_check_sequence(seq, df_actes_out, vol)
    rapports[vol] = seq
    export_interactive_html(seq, vol, os.path.join(OUT_DIR, f'verification_actes_{vol}.html'))

resume = pd.DataFrame([
    {
        'volume': vol,
        'nb_actes_calcules': len(seq),
        'nb_reperes_connus': seq['act_number_table'].notna().sum(),
        'nb_avertissements': (seq['warning'] != '').sum(),
    }
    for vol, seq in rapports.items()
])
resume
